In [ ]:
!pip install sklearn-crfsuite

In [ ]:
import os
import re
import urllib.request
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict, Counter
from sklearn.metrics import classification_report

!wget -P /kaggle/working/ https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/train.mypos-ver3.txt
!wget -P /kaggle/working/ https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.txt
Train_FILE = "train.mypos-ver3.txt"
Train_COL_FILE = "train.nopipe.col"
openTEST_RAW_FILE = "otest.1k.txt"
openTEST_COL_FILE = "otest.nopipe.col"

In [ ]:
# ==========================================
# Line-to-column Converter
# ==========================================
def convert_line_to_col_subwords(input_file, output_file):
    total_sentences = 0
    total_tokens = 0
    
    with open(input_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            line_str = line.strip()
            if not line_str or line_str.startswith("<"):
                continue
            
            raw_tokens = line_str.split()
            sentence_has_words = False
            
            for token in raw_tokens:
                if '/' not in token:
                    continue
                
                # 1. Split token on '|' first
                sub_parts = token.split('|')
                
                # 2. Determine default fallback tag from the last piece with a '/'
                fallback_tag = None
                for part in reversed(sub_parts):
                    if '/' in part:
                        _, fallback_tag = part.rsplit('/', 1)
                        fallback_tag = fallback_tag.strip()
                        break
                
                if not fallback_tag:
                    continue
                
                # 3. Extract word and tag for each subword piece
                for part in sub_parts:
                    part = part.strip()
                    if not part:
                        continue
                    
                    if '/' in part:
                        sub_word, sub_tag = part.rsplit('/', 1)
                        sub_word, sub_tag = sub_word.strip(), sub_tag.strip()
                    else:
                        sub_word, sub_tag = part, fallback_tag
                    
                    if sub_word and sub_tag:
                        outfile.write(f"{sub_word}\t{sub_tag}\n")
                        total_tokens += 1
                        sentence_has_words = True
            
            if sentence_has_words:
                outfile.write("\n")
                total_sentences += 1
                
    print(f"Converted {input_file} -> {output_file}")
    print(f"Total Sentences: {total_sentences}, Total Subword Tokens: {total_tokens}")

# Run conversion
convert_line_to_col_subwords(Train_FILE, Train_COL_FILE)
convert_line_to_col_subwords(openTEST_RAW_FILE, openTEST_COL_FILE)

In [ ]:
# ==========================================
# Load the generated column dataset
# ==========================================
def load_col_file(filepath):
    sentences = []
    current_sentence = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue
            parts = line.split('\t')
            if len(parts) == 2:
                current_sentence.append((parts[0], parts[1]))
                
    if current_sentence:
        sentences.append(current_sentence)
    return sentences

print("Loading column data...")

sentences = load_col_file(Train_COL_FILE)
print(f"Loaded {len(sentences)} sentences.")

In [ ]:
# ==========================================
# Subword-Level Feature Extraction Tagger
# ==========================================
def word2features(sent, i):
    word = sent[i][0]
    
    auxiliary_candidates = {'သွား', 'လာ', 'ပြန်', 'ရ', 'ပေး', 'ကြည့်', 'ခဲ့', 'ထား'}
    connective_candidates = {'နဲ့', 'နှင့်', 'ပြီး'}
    demonstrative_candidates = {'ဒီ', 'ထို', 'ယင်း'}
    postposition_set = {'က', 'မှာ', 'ကို', '၏', 'သို့', 'မှ', 'နဲ့', 'နှင့်', 'တွင်း', 'ပြင်'}
    clause_connectors = {'လျှင်', 'မှ', 'တော့', 'နောက်', 'ပြီး', 'ရင်'}
    sentence_endings = {'တယ်', 'သည်', 'မည်', 'ပြီ', 'ပါ', 'ဖူး'}
    
    is_myanmar_digit = any('\u1040' <= char <= '\u1049' for char in word)

    # 1. Base Unigram, Affixes & Tone Features
    features = {
        'U02': word,                         
        'length': len(word),                 
        'is_digit': is_myanmar_digit,
        
        'suffix_1': word[-1:],
        'suffix_2': word[-2:] if len(word) >= 2 else word,
        'suffix_3': word[-3:] if len(word) >= 3 else word,
        'prefix_1': word[:1],
        'prefix_2': word[:2] if len(word) >= 2 else word,
        'prefix_3': word[:3] if len(word) >= 3 else word,
        
        'has_visarga': 'း' in word,            
        'has_asat': '်' in word,               
        
        'is_auxiliary_candidate': word in auxiliary_candidates,
        'is_connective_candidate': word in connective_candidates,
        'is_demonstrative': word in demonstrative_candidates,
    }

    # 2. Contextual Features: Previous Word (i-1)
    if i > 0:
        prev_word = sent[i-1][0]
        features.update({
            'U01': prev_word,                
            'B1_prev_curr': f"{prev_word}_{word}",  
            'prev_suffix_2': prev_word[-2:] if len(prev_word) >= 2 else prev_word,
            'prev_word_plus_aux_cand': f"{prev_word}_{word}" if word in auxiliary_candidates else "",
        })
    else:
        features['BOS'] = True               

    # 3. Contextual Features: Word Two Positions Back (i-2)
    if i > 1:
        features['U00'] = sent[i-2][0]       
        features['T1_prev2_prev1_curr'] = f"{sent[i-2][0]}_{sent[i-1][0]}_{word}"
    else:
        features['BOS-1'] = True

    # 4. Contextual Features: Next Word (i+1)
    if i < len(sent) - 1:
        next_word = sent[i+1][0]
        features.update({
            'U03': next_word,                
            'B2_curr_next': f"{word}_{next_word}",  
            'next_prefix_2': next_word[:2] if len(next_word) >= 2 else next_word,
            
            'is_dem_followed_by_ppm': (word in demonstrative_candidates) and (next_word in postposition_set),
            'is_conn_followed_by_clause': (word == 'ပြီး') and (next_word in clause_connectors),
            'is_aux_followed_by_ending': (word in auxiliary_candidates) and (next_word in sentence_endings),
            
            'curr_cand_plus_next_word': f"{word}_{next_word}" if word in (demonstrative_candidates | connective_candidates) else "",
        })
    else:
        features['EOS'] = True               

    # 5. Contextual Features: Word Two Positions Ahead (i+2)
    if i < len(sent) - 2:
        features['U04'] = sent[i+2][0]       
        features['T2_curr_next1_next2'] = f"{word}_{sent[i+1][0]}_{sent[i+2][0]}"
    else:
        features['EOS+1'] = True

    # 6. Sandwich Trigram (i-1, i, i+1)
    if i > 0 and i < len(sent) - 1:
        features['T3_prev_curr_next'] = f"{sent[i-1][0]}_{word}_{sent[i+1][0]}"

    return features

In [ ]:
print("Extracting features from dataset...")
X_data = [[word2features(s, i) for i in range(len(s))] for s in sentences]
y_data = [[label for word, label in s] for s in sentences]

In [ ]:
# ==========================================
# Train CRF Model
# ==========================================
print("Training CRF Model (this may take 1-2 minutes)...")

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # Lower L1 regularization to allow more features
    c2=0.1,  # Lower L2 regularization
    max_iterations=150,
    all_possible_transitions=True
)

crf.fit(X_data, y_data)
print("\nFinished training successfully!")

In [ ]:
# ==========================================
# Prepare Train/Test Split
# We use a 20% test split from the loaded 42,196 sentences
# ==========================================
train_sents, test_sents = train_test_split(sentences, test_size=0.2, random_state=42)

# Extract features and ground-truth labels for the test sentences
X_test = [[word2features(s, i) for i in range(len(s))] for s in test_sents]
y_test = [[label for word, label in s] for s in test_sents]

# ==========================================
# Test Trained CRF Model
# ==========================================
print("Running CRF predictions on closed-test data...")
y_pred = crf.predict(X_test)
print("Finished testing!\n")

# ==========================================
# Creates 3-column output: Word | Reference_Tag | Model_Tag
# ==========================================
output_lines = []

for sent, pred in zip(test_sents, y_pred):
    for (word, true_tag), pred_tag in zip(sent, pred):
        output_lines.append(f"{word}\t{true_tag}\t{pred_tag}")
    output_lines.append("")  # Blank line separating sentences

RESULT_FILE = "/kaggle/working/ctest.nopipe.col.result"
with open(RESULT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"Results successfully written to: {RESULT_FILE}\n")

# ==========================================
# Preview Output (First 15 lines)
# ==========================================
print("--- Preview of output (`ctest.nopipe.col.result`) ---")
print("Word\t\tRef_Tag\tPred_Tag")
print("-" * 35)
for line in output_lines[:15]:
    print(line)

In [ ]:
# ==========================================
# Testing with Open Test File
# ==========================================
# Load open test column data
print("Loading open-test data...")
test_sentences = load_col_file(openTEST_COL_FILE)
print(f"Loaded {len(test_sentences)} test sentences.")

# Extract features (X_test) and true labels (y_test)
print("Extracting features from open-test dataset...")
X_test = [[word2features(s, i) for i in range(len(s))] for s in test_sentences]
y_test = [[label for word, label in s] for s in test_sentences]

# Predict POS tags using the trained CRF model
print("Running predictions on open-test set...")
y_pred = crf.predict(X_test)

output_lines = []

for sent, pred in zip(test_sentences, y_pred):
    for (word, true_tag), pred_tag in zip(sent, pred):
        output_lines.append(f"{word}\t{true_tag}\t{pred_tag}")
    output_lines.append("")  # Blank line separating sentences

RESULT_FILE = "/kaggle/working/otest.nopipe.col.result"
with open(RESULT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"Results successfully written to: {RESULT_FILE}\n")

# ==========================================
# Preview Output (First 15 lines)
# ==========================================
print("--- Preview of output (`otest.nopipe.col.result`) ---")
print("Word\t\tRef_Tag\tPred_Tag")
print("-" * 35)
for line in output_lines[:15]:
    print(line)

In [ ]:
# ==========================================
# Format Model Output to Line Format
# ==========================================
# File paths
ctest_COL_RESULT = "/kaggle/working/ctest.nopipe.col.result"
ctest_LINE_FILE = "/kaggle/working/ctest.nopipe.col.result.f13.line"
otest_COL_RESULT = "/kaggle/working/otest.nopipe.col.result"
otest_LINE_FILE = "/kaggle/working/otest.nopipe.col.result.f13.line"

def col_to_line_format(input_filepath, output_filepath):
    """
    Reads 3-column output (Word, Reference_Tag, Predicted_Tag),
    extracts (Word, Predicted_Tag), formats as Word/Tag, and joins into line-formatted sentences.
    """
    formatted_sentences = []
    current_sentence = []

    with open(input_filepath, 'r', encoding='utf-8') as infile:
        for line in infile:
            line = line.strip()
            
            if not line:
                # End of sentence boundary
                if current_sentence:
                    formatted_sentences.append(" ".join(current_sentence))
                    current_sentence = []
                continue
            
            parts = line.split('\t')
            # Extract column 1 (word) and column 3 (predicted tag)
            if len(parts) >= 3:
                word, _, pred_tag = parts[0], parts[1], parts[2]
            elif len(parts) == 2:
                word, pred_tag = parts[0], parts[1]
            else:
                continue

            # Convert to Word/Tag 
            current_sentence.append(f"{word}/{pred_tag}")

    # Catch last sentence if file doesn't end with a trailing newline
    if current_sentence:
        formatted_sentences.append(" ".join(current_sentence))

    # Save to file
    with open(output_filepath, 'w', encoding='utf-8') as outfile:
        outfile.write("\n".join(formatted_sentences) + "\n")

    print(f"Successfully processed {len(formatted_sentences)} sentences into line format.")
    print(f"Saved to: {output_filepath}\n")

col_to_line_format(ctest_COL_RESULT, ctest_LINE_FILE)
col_to_line_format(otest_COL_RESULT, otest_LINE_FILE)
# ==========================================
# Preview Output
# ==========================================
print("--- Head of generated ctest-line-formatted file ---")
with open(ctest_LINE_FILE, 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(f.readline().strip())

print("--- Head of generated otest-line-formatted file ---")
with open(otest_LINE_FILE, 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(f.readline().strip())

In [ ]:
# ==========================================
# 1. Ground Truth Reference for Evaluation
# ==========================================
def create_subword_raw_reference(raw_input_file, raw_subword_ref_file):
    with open(raw_input_file, 'r', encoding='utf-8') as infile, \
         open(raw_subword_ref_file, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            line_str = line.strip()
            if not line_str or line_str.startswith("<"):
                outfile.write(line)
                continue
            
            raw_tokens = line_str.split()
            subword_tokens = []
            
            for token in raw_tokens:
                if '/' not in token:
                    continue
                
                sub_parts = token.split('|')
                fallback_tag = None
                for part in reversed(sub_parts):
                    if '/' in part:
                        _, fallback_tag = part.rsplit('/', 1)
                        fallback_tag = fallback_tag.strip()
                        break
                
                for part in sub_parts:
                    part = part.strip()
                    if not part:
                        continue
                    if '/' in part:
                        sub_w, sub_t = part.rsplit('/', 1)
                        subword_tokens.append(f"{sub_w.strip()}/{sub_t.strip()}")
                    elif fallback_tag:
                        subword_tokens.append(f"{part}/{fallback_tag}")
                        
            outfile.write(" ".join(subword_tokens) + "\n")

openTEST_nopipe = "openTEST_nopipe.txt"
create_subword_raw_reference(openTEST_RAW_FILE, openTEST_nopipe)

In [ ]:
# ==========================================
# Evaluates POS tagger predictions against reference annotations at the subword (no-pipe) level. 
# ==========================================
def evaluate_pos_tagger_nopipe(ref_path, pred_path):
    y_true = []
    y_pred = []
    
    total_words = 0
    correct_words = 0
    total_ref_tags = 0
    total_pred_tags = 0
    
    # Error trackers
    error_pairs_with_words = defaultdict(Counter)
    word_error_counts = defaultdict(lambda: defaultdict(int))
    
    with open(ref_path, 'r', encoding='utf-8') as r_file, \
         open(pred_path, 'r', encoding='utf-8') as p_file:
        
        for line_num, (ref_line, pred_line) in enumerate(zip(r_file, p_file), 1):
            ref_line = ref_line.strip()
            pred_line = pred_line.strip()
            
            if not ref_line or not pred_line:
                continue
            
            raw_ref_tokens = ref_line.split()
            pred_tokens = pred_line.split()
            
            # 1. Clean & expand any compound pipe markers in reference line if present
            clean_ref_tokens = []
            for token in raw_ref_tokens:
                if '|' in token:
                    parts = token.split('|')
                    # Find default fallback tag from last part containing '/'
                    fallback_tag = 'n'
                    for p in reversed(parts):
                        if '/' in p:
                            fallback_tag = p.rsplit('/', 1)[-1].strip()
                            break
                    for p in parts:
                        p = p.strip()
                        if not p:
                            continue
                        if '/' in p:
                            clean_ref_tokens.append(p)
                        else:
                            clean_ref_tokens.append(f"{p}/{fallback_tag}")
                else:
                    clean_ref_tokens.append(token)
            
            ref_tokens = clean_ref_tokens
            
            # Track raw tag counts for Precision & Recall calculation
            total_ref_tags += sum(1 for t in ref_tokens if '/' in t)
            total_pred_tags += sum(1 for t in pred_tokens if '/' in t)
            
            # 2. Align token lengths per line
            if len(ref_tokens) != len(pred_tokens):
                min_len = min(len(ref_tokens), len(pred_tokens))
                ref_tokens = ref_tokens[:min_len]
                pred_tokens = pred_tokens[:min_len]
            
            # 3. Extract and compare subword tags
            for ref_tok, pred_tok in zip(ref_tokens, pred_tokens):
                if '/' not in ref_tok or '/' not in pred_tok:
                    continue
                
                ref_word, true_tag = ref_tok.rsplit('/', 1)
                pred_word, pred_tag = pred_tok.rsplit('/', 1)
                
                ref_word = ref_word.strip()
                true_tag = true_tag.strip()
                pred_tag = pred_tag.strip()
                
                y_true.append(true_tag)
                y_pred.append(pred_tag)
                total_words += 1
                
                if true_tag == pred_tag:
                    correct_words += 1
                else:
                    error_pairs_with_words[(true_tag, pred_tag)][ref_word] += 1
                    word_error_counts[ref_word][f"{true_tag} -> {pred_tag}"] += 1

    # Metrics calculation
    accuracy = (correct_words / total_words * 100) if total_words > 0 else 0
    precision = (correct_words / total_pred_tags * 100) if total_pred_tags > 0 else 0
    recall = (correct_words / total_ref_tags * 100) if total_ref_tags > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    print("================ Step 8 Evaluation Results ================")
    print(f"Total Words Evaluated : {total_words}")
    print(f"Correct Word/Tag Pairs: {correct_words}")
    print(f"Word Tagging Accuracy : {accuracy:.2f}%")
    print(f"Precision             : {precision:.2f}%")
    print(f"Recall                : {recall:.2f}%")
    print(f"F1-Score              : {f1:.2f}%")
    print("===========================================================")

    # 1. Per-Tag Performance Report
    if y_true and y_pred:
        print("\n================ 1. PER-TAG PERFORMANCE REPORT ================")
        print(classification_report(y_true, y_pred, digits=3))
        
        # 2. Top Tag Misclassifications
        print("\n================ 2. TOP 10 TAG MISCLASSIFICATIONS ================")
        print(f"{'Actual Tag':<11} | {'Predicted As':<13} | {'Error Count':<12} | Top Words (Word: Count)")
        print("-" * 85)
        
        sorted_tag_errors = sorted(
            error_pairs_with_words.items(),
            key=lambda item: sum(item[1].values()),
            reverse=True
        )[:10]
        
        for (true_t, pred_t), word_counts in sorted_tag_errors:
            total_err = sum(word_counts.values())
            top_w = ", ".join([f"{w} ({c})" for w, c in word_counts.most_common(5)])
            print(f"{true_t:<11} | {pred_t:<13} | {total_err:<12} | {top_w}")
            
        # 3. Top Subwords with Most Errors
        print("\n================ 3. TOP 10 SUBWORDS WITH MOST ERRORS ================")
        print(f"{'Subword':<20} | {'Errors':<8} | Main Confusion (True -> Pred)")
        print("-" * 60)
        
        sorted_word_errors = sorted(
            word_error_counts.items(),
            key=lambda item: sum(item[1].values()),
            reverse=True
        )[:10]
        
        for word, confusions in sorted_word_errors:
            total_err = sum(confusions.values())
            main_confusion = max(confusions.items(), key=lambda x: x[1])[0]
            print(f"{word:<20} | {total_err:<8} | {main_confusion}")

    return accuracy

# Run Step 8 Evaluation
evaluate_pos_tagger_nopipe(openTEST_nopipe, otest_LINE_FILE)